In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

import os
OpenAI.api_key=os.getenv("OPENAI_API_KEY")

In [3]:
load_dotenv()

client = OpenAI()
models = client.models.list()


In [ ]:
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a assistant which informs about temperature."},
        {"role": "user", "content": "Hey there"}
    ]
)

print(completion.choices[0].message)

In [12]:
# Example dummy function hard coded to return the same weather
# In production, this could be your backend API or an external API
import requests
def get_current_weather(location):
    """Get the current weather in a given location"""

    url = "https://elevation-weather-api.p.rapidapi.com/data"

    querystring = {"query":location}

    headers = {
        "x-rapidapi-key": "3ab9cd3edcmsh321cb5e842b4be1p1e2a51jsn9cf41ca1a28a",
        "x-rapidapi-host": "elevation-weather-api.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    response = requests.get(url, headers=headers, params=querystring)

    print(response.json())
  
    return response.json()

In [13]:
response=get_current_weather('Bangalore')

{'success': True, 'location': 'Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, India', 'coordinates': {'latitude': 12.9767936, 'longitude': 77.590082}, 'elevation': {'meters': 919, 'feet': 3015.09}, 'weather': {'temperature_c': 24.6, 'windspeed_kmh': 12.8, 'wind_direction': 261, 'condition': 'Light drizzle', 'time': '2026-06-14T11:30'}}


In [14]:
response

{'success': True,
 'location': 'Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, India',
 'coordinates': {'latitude': 12.9767936, 'longitude': 77.590082},
 'elevation': {'meters': 919, 'feet': 3015.09},
 'weather': {'temperature_c': 24.6,
  'windspeed_kmh': 12.8,
  'wind_direction': 261,
  'condition': 'Light drizzle',
  'time': '2026-06-14T11:30'}}

In [ ]:
functions = [
        {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. Bengaluru, KA",
                    },
                    
                },
                "required": ["location"],
            },
        }
    ]

In [16]:
functions

[{'name': 'get_current_weather',
  'description': 'Get the current weather in a given location',
  'parameters': {'type': 'object',
   'properties': {'location': {'type': 'string',
     'description': 'The city and state, e.g. San Francisco, CA'}},
   'required': ['location']}}]

In [18]:
user_message="Hi There"
messages=[]

messages.append({"role": "user", "content":user_message})

completion=client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=
       messages
    
)

print(completion.choices[0].message)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [17]:
user_message="What is the temperature of Bangalore"

messages.append({"role": "user", "content": user_message})

completion=client.chat.completions.create(
    messages=messages,
    functions=functions
    
)

messages

NameError: name 'messages' is not defined

In [ ]:
completion

In [ ]:
print(completion.choices[0].message)

In [ ]:
response=completion.choices[0].message

In [ ]:
response

In [ ]:
function_name=response['function_call']['name']
print(function_name)

In [ ]:

import json
location=eval(response['function_call']['arguments'])['location']
print(location)

In [ ]:
# Step 4: send the info on the function call and function response to GPT
messages.append(response)  # extend conversation with assistant's reply
messages.append(
    {
        "role": "function",
        "name": function_name,
        "content": location,
    }
)

In [ ]:
messages

In [ ]:
# extend conversation with function response
second_response = client.chat.completions.create(
    model="gpt-3.5-turbo-0613",
    messages=messages,
    functions=functions
)  # get a new response from GPT where it can see the function response

In [ ]:
print(second_response.choices[0].message)

In [ ]:
second_response